# Tipado y modelos de datos con Pydantic

**Unidad 1 · Sesión 2**

Anotaciones de tipo, análisis estático y validación de datos en Python.

## Contenido

| Sección | Tema |
|---|---|
| 0 | Preparación |
| 1 | Anotaciones de tipo y colecciones |
| 2 | Análisis estático con mypy |
| Ejemplo | Descuentos y precisión del tipo de retorno |
| 3 | Modelos con Pydantic |
| 4 | Restricciones y errores de validación |
| 5 | Validadores de campos |
| 6 | Configuración |
| Ejemplo | Registro de usuarios |
| 7 | Modelos anidados y serialización |
| 8 | Módulo reutilizable |
| 9 | Procesamiento de registros |
| 10 | Una API pequeña con FastAPI y Pydantic |

## Objetivos

- Anotar funciones y colecciones e interpretar diagnósticos de mypy.
- Definir modelos con restricciones y validadores.
- Localizar errores en datos de entrada.
- Serializar modelos anidados y utilizarlos desde un módulo.
- Observar la validación de una petición a una API.

## 0. Preparación

>**mypy** es un verificador estático: analiza el código y contrasta las operaciones
con sus tipos declarados. **Pydantic** es una biblioteca que utiliza anotaciones
para validar y convertir datos durante la ejecución.

Ejecuta la siguiente celda en Colab o Jupyter. Desde terminal puedes instalar las
mismas dependencias con `python -m pip install -r requirements.txt`.

In [1]:
%pip install -q pydantic==2.12.5 mypy==1.19.1 email-validator==2.3.0 fastapi==0.141.1 httpx2==2.13.1


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: /Users/alondra.matos/Documents/Github_IA/Inteligencia_Artificial_Ejercicios/Programacion_para_IA/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from importlib.metadata import version

assert sys.version_info >= (3, 12), "Use Python 3.12 or newer"
print(f"Python: {sys.version.split()[0]}")
print(f"Pydantic: {version('pydantic')}")
print(f"mypy: {version('mypy')}")

Python: 3.12.4


Pydantic: 2.12.5
mypy: 1.19.1


## 1. Del comportamiento al contrato

Una anotación indica el tipo esperado. En `score: float`, `score` es el nombre y
`float` el tipo. Después de los parámetros, `-> str` describe el valor de retorno.
La implementación sigue siendo necesaria: los tipos no calculan el resultado.

**Las anotaciones no validan ni convierten automáticamente los argumentos cuando
Python ejecuta una función.**

In [3]:
def repeat_text(text: str, times: int) -> str:
    return text * times


print(repeat_text("hello ", 2))
# Intentional mismatch: Python executes this even though the annotation says str.
print(repeat_text(3, 2))
assert repeat_text(3, 2) == 6

hello hello 
6


### Análisis estático y ejecución

Las anotaciones describen los tipos esperados de variables, parámetros y retornos.
Mypy utiliza esa información para detectar operaciones incompatibles sin ejecutar
el programa. Las comprobaciones con `assert` verifican resultados concretos cuando
Python sí ejecuta el código.

Estas herramientas tienen alcances diferentes. Una función puede superar el
análisis estático y recibir después un dato externo inválido. El siguiente ejemplo
analiza con mypy la misma llamada que Python aceptó en la celda anterior.

Aquí usamos `mypy_api.run` para analizar texto sin crear un archivo: `-c` pasa
ese texto. `--strict` añade reglas, como exigir funciones anotadas;
`--no-incremental` evita reutilizar resultados de la caché. Más adelante
veremos el comando habitual
sobre un archivo `.py`.

In [4]:
from mypy import api as mypy_api

repeat_source = """def repeat_text(text: str, times: int) -> str:
    return text * times

result = repeat_text(3, 2)
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", repeat_source]
)
print(stdout)
assert exit_status == 1
assert "arg-type" in stdout

runtime_result = repeat_text(3, 2)
print(f"Runtime result: {runtime_result!r}")
assert runtime_result == 6

<string>:4: error: Argument 1 to "repeat_text" has incompatible type "int"; expected "str"  [arg-type]
Found 1 error in 1 file (checked 1 source file)

Runtime result: 6


### Colecciones y valores que pueden faltar

`list[float]` describe una lista de números; `dict[str, int]`, un diccionario de
claves de texto y valores enteros. `float | None` permite un número o ausencia.
No significa «cero». Un argumento con `= None` además puede omitirse en la llamada.

In [5]:
scores: list[float] = [0.0, 0.5, 1.0]
counts: dict[str, int] = {"positive": 2, "neutral": 1}
coordinates: tuple[float, float] = (20.97, -89.62)


def mean_score(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


assert mean_score([]) is None
assert mean_score([0.0]) == 0.0
print(mean_score(scores))


def resolve_threshold(threshold: float | None = None) -> float:
    if threshold is None:
        return 0.5
    return threshold


assert resolve_threshold() == 0.5  # The argument is omitted.
assert resolve_threshold(None) == 0.5  # None is passed explicitly.
assert resolve_threshold(0.0) == 0.0  # Zero is a value, not absence.

0.5


### Ejercicio 1 · Anotar sin cambiar el comportamiento

Anota `select_scores`: recibe una lista de floats y un umbral float; devuelve
una lista de floats. Conserva los valores iguales al umbral. Comprueba lista vacía,
umbral cero y `[0.5, 0.9, 1.0]` con umbral 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [6]:
"""
Predicción: una lista vacía dará []; con umbral cero se conservará 0.0, y con
umbral 0.9 quedarán [0.9, 1.0]. El valor igual al umbral debe conservarse.
Explicación: las anotaciones describen entradas y salida sin cambiar la
comparación >= ni convertir los valores. Los assert comprueban el comportamiento.
"""
def select_scores(values: list[float], threshold: float) -> list[float]:
    return [value for value in values if value >= threshold]


assert select_scores([], 0.9) == []
assert select_scores([0.0, 0.5, 1.0], 0.0) == [0.0, 0.5, 1.0]
assert select_scores([0.5, 0.9, 1.0], 0.9) == [0.9, 1.0]
assert select_scores([0.9], 0.9) == [0.9]
print("Empty:", select_scores([], 0.9))
print("Zero threshold:", select_scores([0.0, 0.5, 1.0], 0.0))
print("Threshold 0.9:", select_scores([0.5, 0.9, 1.0], 0.9))


Empty: []
Zero threshold: [0.0, 0.5, 1.0]
Threshold 0.9: [0.9, 1.0]


## 2. Leer un error del verificador

Esta celda analiza otra cadena de código sin ejecutarla ni crear archivos.
`mypy_api.run` devuelve salida, errores internos y código de salida. El código
1 es esperado aquí: hemos
introducido dos errores deliberados.

Lee el mensaje en este orden: ubicación, operación, tipo recibido y tipo esperado.
No cambies un tipo para callar al verificador: revisa qué comportamiento necesitas.

In [7]:
broken_source = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold("0.9", 0.8)

def display_score(score: float | None) -> str:
    return f"{score * 100:.1f}%"
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", broken_source]
)
print(stdout)
assert exit_status == 1
assert "arg-type" in stdout
assert "operator" in stdout

<string>:4: error: Argument 1 to "apply_threshold" has incompatible type "str"; expected "float"  [arg-type]
<string>:7: error: Unsupported operand types for * ("None" and "int")  [operator]
<string>:7: note: Left operand is of type "float | None"
Found 2 errors in 1 file (checked 1 source file)



### Ejercicio 2 · Corregir el contrato

Corrige los dos problemas de `broken_source`: convierte explícitamente el texto
a float y atiende `None` antes de multiplicar. Trabaja en otra cadena para conservar
el ejemplo defectuoso. Ejecuta mypy y comprueba código de salida 0. ¿Por qué convertir
con `float` todavía puede fallar en tiempo de ejecución?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [8]:
"""
Predicción: convertir explícitamente "0.9" a float y tratar None antes de
multiplicar eliminará los dos diagnósticos: mypy terminará con código 0.
Explicación: la rama de None devuelve un mensaje sin calcular un porcentaje.
Después de esa rama, mypy sabe que score es float. float(texto) todavía puede
lanzar ValueError si el contenido no representa un número; el tipo str no
garantiza un contenido numérico. No modificamos broken_source.
"""
fixed_source = """def apply_threshold(score: float, threshold: float) -> bool:
    return score >= threshold

result = apply_threshold(float("0.9"), 0.8)

def display_score(score: float | None) -> str:
    if score is None:
        return "No score"
    return f"{score * 100:.1f}%"

assert result is True
assert display_score(None) == "No score"
assert display_score(0.0) == "0.0%"
assert display_score(0.9) == "90.0%"
"""
fixed_stdout, fixed_stderr, fixed_exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", fixed_source]
)
print(fixed_stdout or fixed_stderr)
assert fixed_exit_status == 0, fixed_stderr or fixed_stdout
assert 'result = apply_threshold("0.9", 0.8)' in broken_source

fixed_namespace: dict[str, object] = {}
exec(fixed_source, fixed_namespace)
assert fixed_namespace["result"] is True

try:
    float("not a number")
except ValueError as error:
    print(f"Expected conversion error: {error}")
else:
    raise AssertionError("Invalid numeric text should fail at runtime")


Success: no issues found in 1 source file

Expected conversion error: could not convert string to float: 'not a number'


### Precisión del tipo de retorno

El tipo de retorno determina qué operaciones puede comprobar el verificador.
`Iterable[float]` garantiza que los valores pueden recorrerse, pero no garantiza
`len()` ni acceso por índice.

La función siguiente devuelve una lista y declara `list[float]` como retorno.
`Iterable` describe la entrada sin exigir que sea una lista. Su funcionamiento
se estudiará con detalle en la próxima sesión.

In [9]:
from collections.abc import Iterable


def calculate_discounts(items: Iterable[float], discount: float) -> list[float]:
    return [item * (1 - discount) for item in items]


prices = [100.0, 200.0, 300.0]
discounted_prices = calculate_discounts(prices, 0.2)
print(len(discounted_prices))
print(discounted_prices[0])
assert discounted_prices == [80.0, 160.0, 240.0]

3
80.0


Ahora declaramos `Iterable[float]` como retorno, aunque la implementación todavía
construye una lista. Python permite `len()` y el índice porque observa el objeto real;
mypy los rechaza porque solo puede garantizar lo que declara el contrato.

In [10]:
iterable_return_source = """from collections.abc import Iterable

def calculate_discounts(
    items: Iterable[float], discount: float
) -> Iterable[float]:
    return [item * (1 - discount) for item in items]

discounted_prices = calculate_discounts([100.0, 200.0], 0.2)
count = len(discounted_prices)
first = discounted_prices[0]
"""
stdout, stderr, exit_status = mypy_api.run(
    ["--strict", "--no-incremental", "-c", iterable_return_source]
)
print(stdout)
assert exit_status == 1

runtime_namespace: dict[str, object] = {}
exec(iterable_return_source, runtime_namespace)
assert runtime_namespace["count"] == 2
assert runtime_namespace["first"] == 80.0

<string>:9: error: Argument 1 to "len" has incompatible type "Iterable[float]"; expected "Sized"  [arg-type]
<string>:10: error: Value of type "Iterable[float]" is not indexable  [index]
Found 2 errors in 1 file (checked 1 source file)



### Evitar `Any` como respuesta automática

`Any` permite saltarse muchas comprobaciones. Si un dato crudo puede ser cualquier
objeto, `object` obliga a comprobar su tipo antes de usar operaciones específicas.
Con `isinstance` podemos acotar el tipo. En los modelos, `Literal` limitará los
valores permitidos. Hoy no necesitamos genéricos avanzados ni protocolos.

In [11]:
def describe_input(value: object) -> str:
    if isinstance(value, str):
        return f"Text with {len(value)} characters"
    return "Not a text value"


print(describe_input("hello"))
print(describe_input(None))

Text with 5 characters
Not a text value


## 3. Un modelo de datos también es una clase

`BaseModel` es la clase base de Pydantic. Al heredar de ella y declarar campos,
obtenemos construcción, validación y serialización. Una instancia sigue teniendo
atributos: `prediction.score`, en lugar de `record["score"]`.

Comenzamos con un modelo deliberadamente pequeño para observar sus límites.

In [12]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


class BasicPrediction(BaseModel):
    text: str
    score: float


basic_prediction = BasicPrediction.model_validate({"text": "Good service", "score": "0.9"})
print(basic_prediction)
print(type(basic_prediction.score))
assert basic_prediction.score == 0.9
# A float annotation alone does not constrain the numerical range.
print(BasicPrediction(text="Outside the range", score=1.5))

text='Good service' score=0.9
<class 'float'>
text='Outside the range' score=1.5


### Conversión y modo estricto

En modo habitual, Pydantic acepta algunas conversiones, como `"0.9"` a `0.9`.
**Convertir** no es lo mismo que **rechazar todo valor cuyo tipo original difiera**.
El modo estricto restringe conversiones; las restricciones de rango son otra regla.

Decidiremos el contrato según la fuente: un archivo puede contener números como
texto, mientras que una configuración interna puede exigir enteros reales.

In [13]:
try:
    BasicPrediction.model_validate({"text": "Good service", "score": "0.9"}, strict=True)
except ValidationError as error:
    print(error.errors()[0]["type"])
else:
    raise AssertionError("Strict validation should reject the string score")

float_type


## 4. Campos restringidos y errores útiles

`Literal` enumera etiquetas válidas; `Field` añade límites. `Annotated` combina
un tipo con metadatos: el alias `Score` se puede reutilizar sin repetir sus reglas.
`source: str | None = None` permite ausencia y también omitir el campo.

`extra="forbid"` rechaza claves desconocidas, como `scroe`; de otro modo un error
de escritura podría pasar inadvertido. Este primer modelo aún no normaliza texto.

In [14]:
from typing import Annotated, Literal

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class ConstrainedPrediction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None


constrained = ConstrainedPrediction(text="Good", label="positive", score=0.0)
assert constrained.source is None
print(constrained)

text='Good' label='positive' score=0.0 source=None


In [15]:
invalid_record = {"text": "", "label": "unknown", "score": 1.2, "scroe": 0.8}
try:
    ConstrainedPrediction.model_validate(invalid_record)
except ValidationError as error:
    for detail in error.errors():
        print(detail["loc"], detail["type"], detail["msg"])
else:
    raise AssertionError("The record should be rejected")

('text',) string_too_short String should have at least 1 character
('label',) literal_error Input should be 'positive', 'neutral' or 'negative'
('score',) less_than_equal Input should be less than or equal to 1
('scroe',) extra_forbidden Extra inputs are not permitted


Cada entrada de `errors()` describe un problema: `loc` indica el campo, `type`
su categoría y `msg` una explicación. Un registro puede producir varios errores.
La función que procesa un lote debe conservarlos, sin confundir número de errores
con número de registros rechazados.

### Ejercicio 3 · Ausente, nulo e inválido

Prueba registros que omitan `source`, que usen `source=None`, que omitan `score`
y que usen `score=None`. Predice cuáles se aceptan. Comprueba también límites 0 y 1.
Explica por qué `str | None` sin `= None` permitiría nulo pero seguiría siendo obligatorio.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [16]:
"""
Predicción: source omitido y source=None serán aceptados porque su tipo permite
None y tiene valor predeterminado. score omitido dará missing; score=None dará
float_type. Los límites 0 y 1 serán válidos porque ge y le incluyen los extremos.
Explicación: str | None permite un valor nulo, pero no establece un valor por
defecto. Sin = None, Pydantic sigue exigiendo que la clave esté presente.
"""
required_record = {"text": "Good", "label": "positive", "score": 0.5}
without_source = ConstrainedPrediction.model_validate(required_record)
with_null_source = ConstrainedPrediction.model_validate({**required_record, "source": None})
assert without_source.source is None
assert with_null_source.source is None
assert without_source == with_null_source
print("Omitted or null source:", without_source.source, with_null_source.source)

missing_score = {key: value for key, value in required_record.items() if key != "score"}
score_failures = [
    (missing_score, "missing"),
    ({**required_record, "score": None}, "float_type"),
]
for candidate, expected_type in score_failures:
    try:
        ConstrainedPrediction.model_validate(candidate)
    except ValidationError as error:
        details = error.errors()
        assert len(details) == 1
        assert details[0]["loc"] == ("score",)
        assert details[0]["type"] == expected_type
        print(details[0]["loc"], details[0]["type"])
    else:
        raise AssertionError("Missing or null score should be rejected")

for boundary in (0.0, 1.0):
    boundary_prediction = ConstrainedPrediction.model_validate({**required_record, "score": boundary})
    assert boundary_prediction.score == boundary

# This model demonstrates nullable but required, without a default.
class RequiredNullableSource(BaseModel):
    source: str | None


assert RequiredNullableSource.model_validate({"source": None}).source is None
try:
    RequiredNullableSource.model_validate({})
except ValidationError as error:
    assert error.errors()[0]["loc"] == ("source",)
    assert error.errors()[0]["type"] == "missing"
    print("Nullable without a default:", error.errors()[0]["type"])
else:
    raise AssertionError("A nullable field without a default is still required")


Omitted or null source: None None
('score',) missing
('score',) float_type
Nullable without a default: missing


## 5. Validadores: reglas que necesitan código

Queremos aceptar `" POSITIVE "` y rechazar textos de solo espacios. Antes de validar
los campos, quitaremos espacios externos y convertiremos las cadenas a minúsculas.
Es una decisión explícita de este taller, no una limpieza universal para todos los textos.

`@field_validator(..., mode="before")` registra una función que recibe la entrada
cruda. `@classmethod` indica que recibe la clase como `cls`, no una instancia como
`self`. Los decoradores conectan nuestra función con Pydantic; no necesitas crear
un decorador propio. Devuelve el valor transformado o lanza `ValueError`.

Los booleanos tienen un comportamiento numérico en Python; queremos rechazarlos
explícitamente como scores. La validación de rango rechazará también `NaN` e infinito.

In [17]:
class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value

In [18]:
normalized = Prediction.model_validate(
    {"text": "  GOOD SERVICE  ", "label": " POSITIVE ", "score": "0.9"}
)
assert normalized.text == "good service"
assert normalized.label == "positive"
print(normalized)

text='good service' label='positive' score=0.9 source=None


### Ejercicio 4 · Detectar los casos que faltaban

Prueba texto `"   "`, etiqueta `None`, score `True`, `"nan"`, `-0.1` y `1.1`.
Deben rechazarse. Comprueba también que `0.0` sea válido y que la entrada original
no se modifique al normalizarla. ¿Por qué `min_length=1` sin quitar espacios era insuficiente?

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [19]:
"""
Predicción: cada variante tendrá un error en el campo modificado: texto de
espacios, etiqueta nula y scores True, "nan", -0.1 y 1.1 serán rechazados.
0.0 será válido y normalizar la entrada no modificará su diccionario original.
Explicación: min_length=1 por sí solo cuenta los espacios como caracteres.
strip elimina esos espacios antes de comprobar longitud. El validador rechaza
booleanos explícitamente; el rango y allow_inf_nan=False rechazan los demás
scores inválidos. Normalizar produce valores para el modelo, no cambia la entrada.
"""
normalization_input = {
    "text": "  GOOD SERVICE  ",
    "label": " POSITIVE ",
    "score": "0.9",
    "source": " FILE ",
}
input_snapshot = normalization_input.copy()
invalid_variants = [
    ("text", "   ", "string_too_short"),
    ("label", None, "literal_error"),
    ("score", True, "value_error"),
    ("score", "nan", "finite_number"),
    ("score", -0.1, "greater_than_equal"),
    ("score", 1.1, "less_than_equal"),
]
for field, invalid_value, expected_type in invalid_variants:
    candidate = {**normalization_input, field: invalid_value}
    try:
        Prediction.model_validate(candidate)
    except ValidationError as error:
        details = error.errors()
        assert len(details) == 1
        assert details[0]["loc"] == (field,)
        assert details[0]["type"] == expected_type
        print(details[0]["loc"], details[0]["type"], details[0]["msg"])
    else:
        raise AssertionError(f"Invalid {field} was accepted: {invalid_value!r}")

zero_prediction = Prediction.model_validate({**normalization_input, "score": 0.0})
assert zero_prediction.score == 0.0
clean_prediction = Prediction.model_validate(normalization_input)
assert clean_prediction.model_dump() == {
    "text": "good service", "label": "positive", "score": 0.9, "source": "file"
}
assert normalization_input == input_snapshot
assert normalization_input["text"] == "  GOOD SERVICE  "

# Without normalization, the earlier model still accepts whitespace as text.
space_only = ConstrainedPrediction.model_validate(
    {"text": "   ", "label": "positive", "score": 0.0}
)
assert space_only.text == "   "
print("Zero score:", zero_prediction.score)
print("Original input preserved:", normalization_input == input_snapshot)


('text',) string_too_short String should have at least 1 character
('label',) literal_error Input should be 'positive', 'neutral' or 'negative'
('score',) value_error Value error, A boolean is not a score
('score',) finite_number Input should be a finite number
('score',) greater_than_equal Input should be greater than or equal to 0
('score',) less_than_equal Input should be less than or equal to 1
Zero score: 0.0
Original input preserved: True


### Validar asignaciones tiene un alcance concreto

`validate_assignment=True` vuelve a validar al asignar atributos del modelo.
Sin esa opción, construir una instancia válida no garantiza que una asignación
posterior conserve el contrato. Esta opción no vigila cada modificación interna
de una lista: mantenemos las operaciones del lote en funciones conocidas.

In [20]:
try:
    normalized.score = 2.0
except ValidationError as error:
    print(error.errors()[0]["msg"])
assert normalized.score == 0.9

Input should be less than or equal to 1


## 6. Configuración explícita

La configuración también es dato: nombre del lote, umbral de selección y máximo
de registros. Aquí llega como diccionario local, sin variables de entorno ni secretos.

Distingue `model_config` (comportamiento de Pydantic) de `BatchConfig` (parámetros
de nuestra aplicación). El máximo será un entero estricto entre 1 y 1000: no
aceptaremos `"10"` ni `True`. El umbral comparte el contrato de los scores.

In [21]:
class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value

In [22]:
config = BatchConfig.model_validate({"batch_name": " Demo batch ", "threshold": "0.9"})
assert config.batch_name == "Demo batch"
assert config.max_records == 100
print(config)

batch_name='Demo batch' threshold=0.9 max_records=100


### Ejercicio 5 · Configuración antes de procesar

Construye una configuración con nombre `"Practice"`, umbral 0 y máximo 3.
Después comprueba que máximo `"3"`, máximo 0 y nombre `"   "` produzcan errores.
Explica por qué un error de configuración debe detener el lote completo, mientras
que un registro inválido puede conservarse como rechazo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [23]:
"""
Predicción: Practice, umbral 0 y máximo 3 formarán una configuración válida.
Máximo "3" fallará por tipo estricto; máximo 0 fallará por el límite mínimo;
un nombre de espacios quedará vacío y será rechazado por el validador.
Explicación: una configuración inválida afecta las reglas de todo el lote y
debe detenerlo antes de procesar. Un registro inválido afecta una entrada:
podemos conservar sus errores y continuar con las otras bajo reglas válidas.
"""
practice_config_data = {"batch_name": "Practice", "threshold": 0, "max_records": 3}
practice_config = BatchConfig.model_validate(practice_config_data)
assert practice_config.batch_name == "Practice"
assert practice_config.threshold == 0.0
assert practice_config.max_records == 3
print("Valid practice configuration:", practice_config)

invalid_configs = [
    ({"max_records": "3"}, "max_records", "int_type"),
    ({"max_records": 0}, "max_records", "greater_than_equal"),
    ({"batch_name": "   "}, "batch_name", "value_error"),
]
for change, expected_field, expected_type in invalid_configs:
    try:
        BatchConfig.model_validate({**practice_config_data, **change})
    except ValidationError as error:
        details = error.errors()
        assert len(details) == 1
        assert details[0]["loc"] == (expected_field,)
        assert details[0]["type"] == expected_type
        print(details[0]["loc"], details[0]["type"])
    else:
        raise AssertionError(f"Invalid configuration was accepted: {change!r}")


Valid practice configuration: batch_name='Practice' threshold=0.0 max_records=3
('max_records',) int_type
('max_records',) greater_than_equal
('batch_name',) value_error


### Registro de usuarios

El modelo usa `EmailStr` para validar el formato del correo y `SecretStr` para
ocultar la contraseña en su representación habitual. `exclude=True` la excluye
de la serialización. `SecretStr` no cifra ni almacena de forma segura una contraseña.

El rol tiene un valor predeterminado válido. El ejemplo muestra la construcción
del modelo, sus errores de validación y su serialización.

In [24]:
from pydantic import EmailStr, SecretStr


class User(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=2)
    email: EmailStr
    password: SecretStr = Field(exclude=True)
    role: Literal["author", "editor", "admin"] = "author"


user_data = {
    "name": "Alex",
    "email": "alex@example.com",
    "password": "ExamplePassword123",
}
user = User.model_validate(user_data)
print(user)
print(user.model_dump_json(indent=2))
assert user.role == "author"
assert "password" not in user.model_dump()

try:
    User.model_validate({**user_data, "email": "invalid-email"})
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])
else:
    raise AssertionError("Invalid email was accepted")

name='Alex' email='alex@example.com' password=SecretStr('**********') role='author'
{
  "name": "Alex",
  "email": "alex@example.com",
  "role": "author"
}
('email',) value is not a valid email address: An email address must have an @-sign.


Un valor fuera del `Literal` y un campo ausente fallan por motivos distintos.
Comparamos `loc` y `type` para distinguir un valor inválido de un dato requerido.

In [25]:
try:
    User.model_validate({**user_data, "role": "developer"})
except ValidationError as error:
    invalid_role = error.errors()[0]
    print(invalid_role["loc"], invalid_role["type"], invalid_role["msg"])
else:
    raise AssertionError("Invalid role was accepted")

data_without_name = {key: value for key, value in user_data.items() if key != "name"}
try:
    User.model_validate(data_without_name)
except ValidationError as error:
    missing_name = error.errors()[0]
    print(missing_name["loc"], missing_name["type"], missing_name["msg"])
else:
    raise AssertionError("Missing name was accepted")

assert invalid_role["loc"] == ("role",)
assert invalid_role["type"] == "literal_error"
assert missing_name["loc"] == ("name",)
assert missing_name["type"] == "missing"

('role',) literal_error Input should be 'author', 'editor' or 'admin'
('name',) missing Field required


## 7. Modelos anidados y serialización

Un reporte contiene configuración, predicciones y errores: cada componente tiene
su propio modelo. `Field(default_factory=list)` crea una lista por instancia.
Una lista de modelos conserva estructura; no necesitamos diccionarios sin contrato
para pasar resultados entre funciones.

Serializar transforma la representación: `model_dump()` produce un diccionario y
`model_dump_json()` texto JSON. `model_validate_json()` reconstruye y valida desde JSON.
No equivale a escribir automáticamente un archivo ni a guardar en una base de datos.

In [26]:
class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)

In [27]:
first_empty_report = BatchReport(config=config)
second_empty_report = BatchReport(config=config)
first_empty_report.accepted.append(normalized)
assert second_empty_report.accepted == []

example_report = BatchReport(config=config, accepted=[normalized])
payload = example_report.model_dump()
json_text = example_report.model_dump_json(indent=2)
print(json_text)
assert isinstance(payload["accepted"][0], dict)
restored_report = BatchReport.model_validate_json(json_text)
assert restored_report == example_report

{
  "config": {
    "batch_name": "Demo batch",
    "threshold": 0.9,
    "max_records": 100
  },
  "accepted": [
    {
      "text": "good service",
      "label": "positive",
      "score": 0.9,
      "source": null
    }
  ],
  "issues": []
}


### Ejercicio 6 · Ida y vuelta

Crea un reporte vacío, serialízalo a JSON y reconstrúyelo. Comprueba igualdad
y listas vacías. Después cambia el score a 5 en un diccionario obtenido con
`example_report.model_dump()`: al validarlo debe aparecer la ubicación
`("accepted", 0, "score")`. El modelo original debe conservar score 0.9.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [28]:
"""
Predicción: el reporte vacío reconstruido será igual al original y conservará
accepted e issues vacíos. Alterar el score anidado a 5 producirá un error en
("accepted", 0, "score"), sin modificar el score 0.9 del modelo original.
Explicación: model_dump construye una representación con diccionarios y listas
separados del modelo. model_validate y model_validate_json vuelven a validar
la estructura anidada; serializar no permite eludir las restricciones.
"""
roundtrip_empty_report = BatchReport(config=config)
empty_json = roundtrip_empty_report.model_dump_json(indent=2)
restored_empty_report = BatchReport.model_validate_json(empty_json)
assert restored_empty_report == roundtrip_empty_report
assert roundtrip_empty_report.accepted == restored_empty_report.accepted == []
assert roundtrip_empty_report.issues == restored_empty_report.issues == []
assert roundtrip_empty_report.accepted is not restored_empty_report.accepted
assert roundtrip_empty_report.issues is not restored_empty_report.issues
print("Empty report JSON:")
print(empty_json)

corrupted_payload = example_report.model_dump()
corrupted_payload["accepted"][0]["score"] = 5
try:
    BatchReport.model_validate(corrupted_payload)
except ValidationError as error:
    details = error.errors()
    assert len(details) == 1
    assert details[0]["loc"] == ("accepted", 0, "score")
    assert details[0]["type"] == "less_than_equal"
    print(details[0]["loc"], details[0]["type"], details[0]["msg"])
else:
    raise AssertionError("A nested score of 5 should be rejected")

assert corrupted_payload["accepted"][0]["score"] == 5
assert example_report.accepted[0].score == 0.9
assert normalized.score == 0.9
print("Original score preserved:", example_report.accepted[0].score)


Empty report JSON:
{
  "config": {
    "batch_name": "Demo batch",
    "threshold": 0.9,
    "max_records": 100
  },
  "accepted": [],
  "issues": []
}
('accepted', 0, 'score') less_than_equal Input should be less than or equal to 1
Original score preserved: 0.9


## 8. De la notebook a un módulo reutilizable

La siguiente celda reúne las clases anteriores y
las funciones de procesamiento en `lesson_models.py`, el mismo archivo que acompaña
esta notebook en el repositorio. Así Colab funciona sin descargar archivos auxiliares.

**La celda escribe o reemplaza `lesson_models.py` en el directorio de trabajo.**
Si hiciste cambios propios en ese archivo, guárdalos con otro nombre antes de
volver a ejecutarla. `%%writefile` es una instrucción de Jupyter, no sintaxis Python.

Lee primero `process_records`: valida cada entrada, guarda cada problema con su
posición desde cero y no captura excepciones ajenas a validación. Si el lote supera
`max_records`, lanza `ValueError` antes de procesarlo. El umbral selecciona entre
registros válidos; no convierte en inválidas las predicciones con menor score.

Importamos el módulo como `models`: el prefijo distingue sus clases de las
versiones construidas durante la explicación y evita mezclar sus instancias.

In [29]:
%%writefile lesson_models.py
"""Typed contracts for synthetic prediction records; no model is trained here."""

from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

Label = Literal["positive", "neutral", "negative"]
Score = Annotated[float, Field(ge=0, le=1, allow_inf_nan=False)]


class Prediction(BaseModel):
    """Normalize text fields and validate one incoming prediction."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    text: str = Field(min_length=1)
    label: Label
    score: Score
    source: str | None = None

    @field_validator("text", "label", "source", mode="before")
    @classmethod
    def normalize_strings(cls, value: object) -> object:
        if isinstance(value, str):
            return value.strip().lower()
        return value

    @field_validator("score", mode="before")
    @classmethod
    def reject_boolean_score(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a score")
        return value


class BatchConfig(BaseModel):
    """Application settings supplied explicitly, without secrets or services."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)

    batch_name: str = Field(min_length=1)
    threshold: Score = 0.9
    max_records: int = Field(default=100, ge=1, le=1000, strict=True)

    @field_validator("batch_name")
    @classmethod
    def require_nonblank_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Batch name must not be blank")
        return value

    @field_validator("threshold", mode="before")
    @classmethod
    def reject_boolean_threshold(cls, value: object) -> object:
        if isinstance(value, bool):
            raise ValueError("A boolean is not a threshold")
        return value


class ValidationIssue(BaseModel):
    position: int
    field: str
    message: str
    error_type: str


class BatchReport(BaseModel):
    config: BatchConfig
    accepted: list[Prediction] = Field(default_factory=list)
    issues: list[ValidationIssue] = Field(default_factory=list)


def average_score(predictions: list[Prediction]) -> float | None:
    if not predictions:
        return None
    return sum(prediction.score for prediction in predictions) / len(predictions)


def process_records(records: list[object], config: BatchConfig) -> BatchReport:
    """Reject an oversized batch; preserve all validation issues for each row."""
    if len(records) > config.max_records:
        raise ValueError("Batch exceeds max_records")

    report = BatchReport(config=config)
    for position, record in enumerate(records):
        try:
            prediction = Prediction.model_validate(record)
        except ValidationError as error:
            for detail in error.errors():
                report.issues.append(
                    ValidationIssue(
                        position=position,
                        field=".".join(str(part) for part in detail["loc"]) or "<record>",
                        message=detail["msg"],
                        error_type=detail["type"],
                    )
                )
        else:
            report.accepted.append(prediction)
    return report


def select_predictions(report: BatchReport) -> list[Prediction]:
    """Select by confidence; this is not an accuracy measurement."""
    return [
        prediction
        for prediction in report.accepted
        if prediction.score >= report.config.threshold
    ]

Overwriting lesson_models.py


### Comprobar un archivo desde la terminal

Con el módulo ya guardado, el uso habitual es `mypy --strict lesson_models.py`.
El argumento final es el archivo que mypy revisará. No hace falta `-c` porque
el código ya está en un archivo, ni `--no-incremental` para el trabajo cotidiano.

La siguiente celda ejecuta el equivalente mediante el Python de esta notebook
y comprueba que termina sin errores.

In [30]:
import importlib
import subprocess
import lesson_models as models

# Reload after running the file-writing cell again.
models = importlib.reload(models)
from lesson_models import average_score, process_records, select_predictions

result = subprocess.run(
    [sys.executable, "-m", "mypy", "--strict", "lesson_models.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout or result.stderr)
assert result.returncode == 0, result.stderr or result.stdout

Success: no issues found in 1 source file



## 9. Práctica guiada: de entradas crudas a reporte

Estos seis registros fueron escritos para esta lección y se pueden reutilizar con
los materiales del curso. Incluyen dos válidos, un score fuera de rango, una
etiqueta nula, un score ausente y una entrada que no es un diccionario.

Antes de ejecutar: identifica posiciones válidas y rechazadas. Esperamos **2
aceptadas**, **4 posiciones rechazadas**, promedio **0.49** y **1 seleccionada** con
umbral 0.9. No calculamos exactitud: no existen etiquetas de referencia.

In [31]:
raw_records: list[object] = [
    {"text": " Excellent ", "label": " POSITIVE ", "score": "0.98"},
    {"text": "Zero is valid", "label": "neutral", "score": 0.0},
    {"text": "Out of range", "label": "negative", "score": 1.2},
    {"text": "Missing label", "label": None, "score": 0.8},
    {"text": "Missing score", "label": "neutral"},
    None,
]

batch_config = models.BatchConfig(batch_name="Class demo", threshold=0.9, max_records=10)
report = process_records(raw_records, batch_config)
rejected_positions = {issue.position for issue in report.issues}
print(f"Accepted: {len(report.accepted)}")
print(f"Rejected records: {len(rejected_positions)}")
print(f"Average score: {average_score(report.accepted)}")
print(f"Selected: {len(select_predictions(report))}")
for issue in report.issues:
    print(issue.position, issue.field, issue.error_type)

Accepted: 2
Rejected records: 4
Average score: 0.49
Selected: 1
2 score less_than_equal
3 label literal_error
4 score missing
5 <record> model_type


### Ejercicio 7 · Comprobar antes de ampliar

Verifica los resultados esperados del lote. Procesa una lista vacía y otra con
solo `None`: sus promedios deben ser `None`. Prueba un lote mayor que el máximo:
debe producir `ValueError`, sin truncarlo silenciosamente. Usa el módulo de apoyo.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [32]:
"""
Predicción: el lote demostrativo tendrá dos aceptadas, cuatro posiciones
rechazadas {2, 3, 4, 5}, promedio 0.49 y una seleccionada con umbral 0.9.
Los lotes vacío y con solo None tendrán promedio None. Superar max_records
producirá ValueError antes de procesar, sin seleccionar solo una parte del lote.
Explicación: el umbral selecciona entre predicciones válidas; el score cero
sigue siendo válido y cuenta en el promedio. Un rechazo se cuenta por posición,
no por número de errores. Utilizamos las clases y funciones del módulo models.
"""
from math import isclose

assert len(report.accepted) == 2
assert [prediction.score for prediction in report.accepted] == [0.98, 0.0]
assert rejected_positions == {2, 3, 4, 5}
assert {issue.position for issue in report.issues} == {2, 3, 4, 5}
assert len(report.issues) == 4
assert isclose(models.average_score(report.accepted), 0.49)
selected_demo = models.select_predictions(report)
assert len(selected_demo) == 1
assert selected_demo[0].score == 0.98
assert report.accepted[1].score == 0.0

empty_batch_report = models.process_records([], batch_config)
assert empty_batch_report.accepted == []
assert empty_batch_report.issues == []
assert models.average_score(empty_batch_report.accepted) is None
assert models.select_predictions(empty_batch_report) == []

none_batch_report = models.process_records([None], batch_config)
assert none_batch_report.accepted == []
assert models.average_score(none_batch_report.accepted) is None
assert len(none_batch_report.issues) == 1
assert none_batch_report.issues[0].position == 0
assert none_batch_report.issues[0].field == "<record>"
assert models.select_predictions(none_batch_report) == []

at_limit_records: list[object] = [raw_records[0]] * batch_config.max_records
at_limit_report = models.process_records(at_limit_records, batch_config)
assert len(at_limit_report.accepted) == batch_config.max_records
assert at_limit_report.issues == []

oversized_records: list[object] = [raw_records[0]] * (batch_config.max_records + 1)
try:
    models.process_records(oversized_records, batch_config)
except ValueError as error:
    assert str(error) == "Batch exceeds max_records"
    print(f"Expected oversized batch error: {error}")
else:
    raise AssertionError("An oversized batch should fail instead of being truncated")
assert len(oversized_records) == batch_config.max_records + 1
print("Demo checks: 2 accepted, 4 rejected positions, mean 0.49, 1 selected.")
print("Empty and all-invalid means:",
      models.average_score(empty_batch_report.accepted),
      models.average_score(none_batch_report.accepted))


Expected oversized batch error: Batch exceeds max_records
Demo checks: 2 accepted, 4 rejected positions, mean 0.49, 1 selected.
Empty and all-invalid means: None None


### Ejercicio 8 · Actividad independiente

Escribe una función tipada `count_labels(predictions)` que devuelva un diccionario
de conteos por etiqueta. Usa `list[models.Prediction]` como entrada y `dict[str, int]`
como salida. Debe funcionar para lista vacía. Añádela a una copia de tu módulo,
compruébala y ejecuta mypy sobre esa copia. Dentro de ese módulo usa
`list[Prediction]`, porque la clase se define allí; el prefijo `models` se usa
solo desde esta notebook.

Como parte de la misma evidencia, crea un registro con **dos errores**, conserva
ambos y explica por qué cuenta como **un registro rechazado**. Serializa el reporte
y comprueba que se pueda reconstruir.

Predice, escribe, ejecuta y explica. Conserva las comprobaciones en la celda.

In [33]:
print("""Predicción: el lote de ejemplo produce {'positive': 1, 'neutral': 1};
una lista vacía produce {} y repetir una predicción incrementa su etiqueta.
Un nuevo registro con label='unknown' y score=1.5 produce dos errores, pero
solo añade una posición rechazada. El reporte JSON podrá reconstruirse con
los mismos datos, incluidas ambas incidencias.""")


def count_labels(predictions: list[models.Prediction]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for prediction in predictions:
        counts[prediction.label] = counts.get(prediction.label, 0) + 1
    return counts


# Check the notebook function with empty, demo, repeated and negative labels.
assert count_labels([]) == {}
assert count_labels(report.accepted) == {"positive": 1, "neutral": 1}
repeated_predictions = report.accepted + [report.accepted[0]]
assert count_labels(repeated_predictions) == {"positive": 2, "neutral": 1}
negative_prediction = models.Prediction.model_validate(
    {"text": " Negative example ", "label": " NEGATIVE ", "score": 0.7}
)
assert count_labels(repeated_predictions + [negative_prediction]) == {
    "positive": 2, "neutral": 1, "negative": 1,
}

# The preserved module uses its own Prediction and BatchConfig classes.
import practice_models
from pathlib import Path

practice_models = importlib.reload(practice_models)
mypy_result = subprocess.run(
    [sys.executable, "-m", "mypy", "--strict", "--no-incremental", "practice_models.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(mypy_result.stdout or mypy_result.stderr)
assert mypy_result.returncode == 0, mypy_result.stderr or mypy_result.stdout

copy_predictions = [
    practice_models.Prediction.model_validate(prediction.model_dump())
    for prediction in report.accepted
]
assert practice_models.count_labels([]) == {}
assert practice_models.count_labels(copy_predictions) == {"positive": 1, "neutral": 1}
assert practice_models.count_labels(copy_predictions + [copy_predictions[0]]) == {
    "positive": 2, "neutral": 1,
}
copy_negative = practice_models.Prediction.model_validate(negative_prediction.model_dump())
assert practice_models.count_labels(copy_predictions + [copy_negative]) == {
    "positive": 1, "neutral": 1, "negative": 1,
}

# Preserve both issues from one additional record without changing raw_records.
two_error_record = {"text": "Two issues", "label": "unknown", "score": 1.5}
extended_records: list[object] = raw_records + [two_error_record]
copy_config = practice_models.BatchConfig.model_validate(batch_config.model_dump())
exercise8_report = practice_models.process_records(extended_records, copy_config)
added_position = len(raw_records)
two_error_issues = [
    issue for issue in exercise8_report.issues if issue.position == added_position
]
assert len(two_error_issues) == 2
assert {(issue.field, issue.error_type) for issue in two_error_issues} == {
    ("label", "literal_error"), ("score", "less_than_equal"),
}
assert {issue.position for issue in two_error_issues} == {added_position}
assert len(exercise8_report.issues) == len(report.issues) + 2 == 6
assert {issue.position for issue in exercise8_report.issues} == {2, 3, 4, 5, 6}
assert len(exercise8_report.accepted) == 2
assert [prediction.model_dump() for prediction in exercise8_report.accepted] == [
    prediction.model_dump() for prediction in report.accepted
]
assert practice_models.count_labels(exercise8_report.accepted) == {
    "positive": 1, "neutral": 1,
}
for issue in two_error_issues:
    print(issue.position, issue.field, issue.error_type, issue.message)
print(f"Validation issues: {len(exercise8_report.issues)}")
print(f"Rejected records: {len({issue.position for issue in exercise8_report.issues})}")

# Save the complete JSON report and reconstruct it from the saved file.
exercise8_json = exercise8_report.model_dump_json(indent=2)
report_path = Path("exercise8_report.json")
report_path.write_text(exercise8_json + "\n", encoding="utf-8")
reconstructed_report = practice_models.BatchReport.model_validate_json(
    report_path.read_text(encoding="utf-8")
)
assert reconstructed_report == exercise8_report
assert reconstructed_report.model_dump() == exercise8_report.model_dump()
assert len([issue for issue in reconstructed_report.issues
            if issue.position == added_position]) == 2
print(exercise8_json)
print(f"JSON reconstruction passed: {report_path.name}")

print("""Explicación: count_labels recorre predicciones ya validadas y acumula
cada etiqueta con dict.get; para una lista vacía el diccionario permanece vacío.
La anotación describe el contrato, pero la validación la realizan los modelos.
El módulo practice_models.py conserva los contratos originales y añade la función;
sus propias clases se usan para sus operaciones, convirtiendo los datos con
model_dump/model_validate. mypy terminó sin errores sobre esa copia.
Las incidencias de label y score comparten la posición 6: son dos problemas
de un único registro. Por eso el lote contiene 6 incidencias y 5 posiciones
rechazadas. La igualdad tras reconstruir el JSON verifica que se conservaron
la configuración, las predicciones aceptadas y todos los errores.""")


Predicción: el lote de ejemplo produce {'positive': 1, 'neutral': 1};
una lista vacía produce {} y repetir una predicción incrementa su etiqueta.
Un nuevo registro con label='unknown' y score=1.5 produce dos errores, pero
solo añade una posición rechazada. El reporte JSON podrá reconstruirse con
los mismos datos, incluidas ambas incidencias.


Success: no issues found in 1 source file

6 label literal_error Input should be 'positive', 'neutral' or 'negative'
6 score less_than_equal Input should be less than or equal to 1
Validation issues: 6
Rejected records: 5
{
  "config": {
    "batch_name": "Class demo",
    "threshold": 0.9,
    "max_records": 10
  },
  "accepted": [
    {
      "text": "excellent",
      "label": "positive",
      "score": 0.98,
      "source": null
    },
    {
      "text": "zero is valid",
      "label": "neutral",
      "score": 0.0,
      "source": null
    }
  ],
  "issues": [
    {
      "position": 2,
      "field": "score",
      "message": "Input should be less than or equal to 1",
      "error_type": "less_than_equal"
    },
    {
      "position": 3,
      "field": "label",
      "message": "Input should be 'positive', 'neutral' or 'negative'",
      "error_type": "literal_error"
    },
    {
      "position": 4,
      "field": "score",
      "message": "Field required",
      "error_type":

## 10. Una API pequeña con FastAPI y Pydantic

Una API puede recibir JSON cuyos valores no conocemos al escribir el programa.
FastAPI usa el modelo `QuoteRequest` para validar el cuerpo de la petición
antes de llamar a la función. Si los datos son válidos, la función recibe
una instancia con sus campos ya convertidos. `response_model` describe y
valida la respuesta.

`TestClient` enviará peticiones a la aplicación desde esta notebook, sin
iniciar un servidor. La organización de la aplicación como proyecto vendrá
en la sesión de uv.

In [34]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field


class QuoteRequest(BaseModel):
    item_name: str = Field(min_length=1)
    unit_price: float = Field(gt=0)
    quantity: int = Field(ge=1)


class QuoteResponse(BaseModel):
    item_name: str
    unit_price: float
    quantity: int
    total: float


app = FastAPI()


@app.post("/quotes", response_model=QuoteResponse)
def create_quote(quote: QuoteRequest) -> QuoteResponse:
    return QuoteResponse(
        item_name=quote.item_name,
        unit_price=quote.unit_price,
        quantity=quote.quantity,
        total=quote.unit_price * quote.quantity,
    )


client = TestClient(app)

### Una petición válida y otra que requiere conversión

En la segunda petición, `unit_price` llega como texto. Pydantic comprueba
que pueda convertirse a `float`; la función de la ruta recibe el número y
calcula el total. La respuesta JSON contiene un número, no el texto original.

In [35]:
valid_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": 19.5, "quantity": 2},
)
converted_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": "19.5", "quantity": 2},
)

assert valid_response.status_code == 200
assert converted_response.status_code == 200
assert valid_response.json() == converted_response.json()
assert converted_response.json()["total"] == 39.0
print(converted_response.json())

{'item_name': 'Notebook', 'unit_price': 19.5, 'quantity': 2, 'total': 39.0}


### Una petición rechazada

El texto `"nineteen"` no puede convertirse a `float` y una cantidad de
cero incumple la restricción `ge=1`. FastAPI responde con estado HTTP 422
y señala ambos campos. La función `create_quote` no se ejecuta para esta
petición.

In [36]:
invalid_response = client.post(
    "/quotes",
    json={"item_name": "Notebook", "unit_price": "nineteen", "quantity": 0},
)

assert invalid_response.status_code == 422
issues = invalid_response.json()["detail"]
assert {tuple(issue["loc"]) for issue in issues} == {
    ("body", "unit_price"),
    ("body", "quantity"),
}
for issue in issues:
    print(issue["loc"], issue["type"])

['body', 'unit_price'] float_parsing
['body', 'quantity'] greater_than_equal


## Evidencia y criterios

Conserva tu módulo con la función del ejercicio 8,
las comprobaciones y un reporte JSON. Puedes copiar la salida JSON o guardarla
localmente; todavía no se exige una base de datos ni un proyecto empaquetado.

- **Contratos claros:** parámetros y retornos anotados; ausencia de `Any` injustificado.
- **Validación:** restricciones y normalización justificadas; cero válido, faltantes y
  entradas inválidas distinguidos; configuración validada antes de procesar.
- **Errores útiles:** posición, campo y motivo conservados, incluidos varios errores por registro.
- **Serialización:** reconstrucción equivalente del reporte anidado.
- **Verificación:** mypy sin errores en el módulo final y comprobaciones de casos límite.

### Profundización

Compara `model_dump(exclude_none=True)` con la salida habitual. ¿Cuál es la diferencia
entre omitir `source` en la representación y cambiar el contrato del modelo?

## Cierre

Explica por qué una anotación no reemplaza la validación, qué información aporta
un error de Pydantic y qué decisiones de conversión tomaste. La próxima sesión
procesa datos de forma perezosa con iteradores, generadores y context managers.

## Referencias

- [Pydantic: modelos](https://docs.pydantic.dev/latest/concepts/models/).
- [Pydantic: validadores](https://docs.pydantic.dev/latest/concepts/validators/).
- [mypy: primeros pasos](https://mypy.readthedocs.io/en/stable/getting_started.html).
- [FastAPI: cuerpo de petición](https://fastapi.tiangolo.com/tutorial/body/).
- [FastAPI: TestClient](https://fastapi.tiangolo.com/tutorial/testing/).

- [PEP 1: propósito y proceso](https://peps.python.org/pep-0001/).
- [PEP 484: Type Hints](https://peps.python.org/pep-0484/).
